In [ ]:
import torch
import torch.utils.data as data
import torch.nn as nn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

from collections import Counter
import string

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [ ]:
import requests

response = requests.get('https://data-api.coindesk.com/news/v1/article/list',
    params={"lang":"EN","limit":100,"to_ts":1697283875,"categories":"BTC","api_key":"cafd7368a08751e251a6b2191b4815fb0d762b38b402708cb141a4497d781eb1"},
    headers={"Content-type":"application/json; charset=UTF-8"}
)

json_response = response.json()
req = json_response['Data']
len(req)

100

In [ ]:
req[0]

{'TYPE': '121',
 'ID': 18808119,
 'GUID': 'https://coinedition.com/?p=342932',
 'PUBLISHED_ON': 1697283660,
 'PUBLISHED_ON_NS': None,
 'IMAGE_URL': 'https://resources.cryptocompare.com/news/7/default.png',
 'TITLE': 'There is a 90% Chance of Bitcoin ETF Approval by January: Analyst',
 'SUBTITLE': None,
 'AUTHORS': None,
 'URL': 'https://coinedition.com/there-is-a-90-chance-of-bitcoin-etf-approval-by-january-analyst/',
 'SOURCE_ID': 7,
 'BODY': 'According to the famous Bitcoin investor Lark Davis, several analysts believe there is a 90% chance that Bitcoin ETFs will be approved by January. Davis made the statement via a post on X (formerly Twitter) and was supported by other crypto users with a similar sentiment. One of Davis’ respondents, Mister Crypto, noted that he The post There is a 90% Chance of Bitcoin ETF Approval by January: Analyst appeared first on Coin Edition .',
 'KEYWORDS': 'News|Bitcoin ETF|Market News',
 'LANG': 'EN',
 'UPVOTES': 0,
 'DOWNVOTES': 0,
 'SCORE': 0,
 'SENTI

In [ ]:
df = pd.DataFrame(req)

In [ ]:
len(df)

100

In [ ]:
df['PUBLISHED_ON'] = pd.to_datetime(df['PUBLISHED_ON'], unit='s')

In [ ]:
df['PUBLISHED_ON'][99]

Timestamp('2023-10-13 12:49:29')

In [ ]:
curr_time = pd.to_datetime(df['PUBLISHED_ON'][len(df) - 1])
curr_time

Timestamp('2023-10-13 12:49:29')

In [ ]:
pd.to_datetime(1657730129, unit='s'), pd.to_datetime(1763120589, unit='s')

(Timestamp('2022-07-13 16:35:29'), Timestamp('2025-11-14 11:43:09'))

In [ ]:
import requests

df = pd.DataFrame()

lim = 100
start_date = 1763120589
lan = 'EN'
categ = 'BTC'
api = 'cafd7368a08751e251a6b2191b4815fb0d762b38b402708cb141a4497d781eb1'
year = 365

for _ in range(5 * year):
  # iter back in dates
  response = requests.get('https://data-api.coindesk.com/news/v1/article/list',
      params={"lang":lan,"limit":lim,"to_ts":start_date,"categories":categ,"api_key":api},
      headers={"Content-type":"application/json; charset=UTF-8"}
  )
  json_response = response.json()
  curr_df = pd.DataFrame(json_response['Data'])
  start_date = curr_df['PUBLISHED_ON'][len(curr_df) - 1] - 1

  df = pd.concat([df, curr_df])

In [ ]:
df = df.reset_index(drop=True)

In [137]:
df.to_csv('data_coindesk_base.csv', index=False)

In [ ]:
import csv
df.to_csv('data_coindesk.csv', index=False, quoting=csv.QUOTE_NONNUMERIC)

In [138]:
df['BODY'][0]

'Macroeconomic uncertainties, such as the US federal government\'s longest-ever 43-day shutdown and uncertainty surrounding the Fed\'s interest rate cuts, caused Bitcoin\'s sharp decline. While further declines and bearish rhetoric for Bitcoin increased after the sharp correction in the market, CryptoQuant CEO Ki-young Ju made important statements about the future of BTC. The famous CEO spoke clearly about the bear market, stating that it is unlikely that a bear market will occur unless the BTC price falls below $94,000. At this point, Ki-young Ju stated that the $94,000 price level is a critical and important level to determine whether Bitcoin has entered a bear market. The famous CEO said in his post on X that it is difficult to evaluate the current correction in Bitcoin as a bear trend on its own. Noting that the average purchase price for investors who purchased BTC 6 to 12 months ago was $94,000, Ju said that whether this level will be broken will determine the bear trend. Finally